# Formula 1 Podium Prediction — Milestone 1: Data Engineering Questions

This notebook answers the data engineering questions. It loads the cleaned tables saved by the cleaning notebook (`ms1_cleaning.ipynb`), so run that notebook first.

## Setup & loading the cleaned tables

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

raw_path   = '..\\ACL_9th-F1\\F1\\'        # original Ergast tables
clean_path = '..\\ACL_9th-F1\\F1_clean\\'  # cleaned tables saved by ms1_cleaning.ipynb

# cleaned tables (produced in the cleaning notebook)
results_clean      = pd.read_csv(clean_path + 'results_clean.csv')
races_clean        = pd.read_csv(clean_path + 'races_clean.csv')
circuits_clean     = pd.read_csv(clean_path + 'circuits_clean.csv')
drivers_clean      = pd.read_csv(clean_path + 'drivers_clean.csv')
constructors_clean = pd.read_csv(clean_path + 'constructors_clean.csv')

# raw lookup tables still used by the questions
races        = pd.read_csv(raw_path + 'races.csv')         # race names (Indy 500 filter), year
constructors = pd.read_csv(raw_path + 'constructors.csv')  # constructor names
status       = pd.read_csv(raw_path + 'status.csv')        # status text for statusId


# Data Engineering questions

## Question 1

In [ ]:
import pandas as pd

In [ ]:
results_clean=pd.read_csv('..\\ACL_9th-F1\\F1_clean\\results_clean.csv')
races_clean=pd.read_csv('..\\ACL_9th-F1\\F1_clean\\races_clean.csv')
circuits_clean=pd.read_csv('..\\ACL_9th-F1\\F1_clean\\circuits_clean.csv')


In [ ]:
merged_df = (results_clean.merge(races_clean, on="raceId", how="left")
         .merge(circuits_clean, on="circuitId", how="left"))
 

In [ ]:
merged_df.info()

In [ ]:
circuits_races_results=merged_df.to_csv('..\\ACL_9th-F1\\F1_clean\\circuits_races_results.csv', index=False)

In [ ]:
merged_df.isnull().sum()

In [ ]:
races_clean.info()

In [ ]:
results_clean.info()

In [ ]:


import matplotlib.pyplot as plt
# join: results -> races -> circuits
df = results_clean.merge(races_clean, on="raceId").merge(circuits_clean, on="circuitId")

# front row by grid (grid 0 = pit-lane start, so it is excluded)
df = df[df.grid.isin([1, 2])].copy()
df["podium"] = df.race_position_order <= 3
df["era"] = df.race_year.ge(2014).map({False: "pre-2014", True: "2014+"})

# conversion rate per circuit and era, keeping circuits with enough starts
t = (df.groupby(["circuitId", "country", "era"]).podium
       .agg(starts="size", rate="mean").reset_index())
t = t[t.starts >= 15]

# side-by-side table + rank in each era
w = t.pivot(index=["circuitId", "country"], columns="era", values="rate").dropna()
w["rank_pre"] = w["pre-2014"].rank(ascending=False)
w["rank_post"] = w["2014+"].rank(ascending=False)
print(w.sort_values("rank_post").round(2))
print("Rank correlation:", round(w["pre-2014"].corr(w["2014+"], method="spearman"), 2))

# plot
w[["pre-2014", "2014+"]].sort_values("2014+").plot.barh(figsize=(8, 7))
plt.xlabel("Front-row start -> podium rate")
plt.tight_layout(); plt.show()

In [ ]:
df['country'].value_counts()

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

P = "/mnt/user-data/uploads/"
res = pd.read_csv(P + "results_clean.csv", na_values=r"\N")
races = pd.read_csv(P + "races_clean.csv")
circ = pd.read_csv(P + "circuits_clean.csv")
drivers = pd.read_csv(P + "drivers.csv", na_values=r"\N")      # needs: driverId, nationality

# same merged table as Q1 (but NOT filtered to the front row), plus driver nationality
df = (res.merge(races, on="raceId").merge(circ, on="circuitId")
         .merge(drivers[["driverId", "nationality"]], on="driverId", how="left"))

# nationality (adjective) -> circuit country name used in circuits_clean
nat2country = {
    "British": "UK", "American": "USA", "German": "Germany", "French": "France",
    "Italian": "Italy", "Brazilian": "Brazil", "Spanish": "Spain", "Australian": "Australia",
    "Austrian": "Austria", "Belgian": "Belgium", "Canadian": "Canada", "Mexican": "Mexico",
    "Japanese": "Japan", "Dutch": "Netherlands", "Swiss": "Switzerland", "Swedish": "Sweden",
    "Argentine": "Argentina", "Argentinian": "Argentina", "Hungarian": "Hungary",
    "Russian": "Russia", "South African": "South Africa", "Chinese": "China",
    "Portuguese": "Portugal", "Monegasque": "Monaco", "Indian": "India",
    "Malaysian": "Malaysia", "Turkish": "Turkey",
}
df["home_country"] = df.nationality.map(nat2country)
print("Nationalities with no mapping:", sorted(df.loc[df.home_country.isna(), "nationality"].dropna().unique()))

df["home"] = (df.home_country == df.country).astype(int)
df["podium"] = (df.race_position_order <= 3).astype(int)

# keep real race starts with a real grid slot (grid 0 = pit lane / no slot)
d = df[(df.grid >= 1) & df.nationality.notna()].copy()
d["grid_c"] = d.grid.clip(upper=20)          # cap very large grids

# 1) raw comparison
print("\nRaw podium rate:\n", d.groupby("home").podium.agg(["size", "mean"]).round(3))
print("Mean grid:\n", d.groupby("home").grid.mean().round(2))   # do home drivers start better?

# 2) controlled comparison: podium rate by grid slot, home vs away
by_grid = d.groupby(["grid_c", "home"]).podium.mean().unstack()
print("\nPodium rate by grid slot:\n", by_grid.round(2).head(10))

# 3) logistic regression with a separate effect for every grid slot
m = smf.logit("podium ~ home + C(grid_c)", data=d).fit(disp=0, cov_type="cluster",
                                                     cov_kwds={"groups": d.driverId})
ci = np.exp(m.conf_int().loc["home"])
print("\nHome effect (odds ratio):", round(np.exp(m.params["home"]), 2),
      "| 95% CI:", [round(x, 2) for x in ci], "| p =", round(m.pvalues["home"], 4))

# plot: podium rate by grid for home vs away
by_grid.head(12).plot(marker="o", figsize=(8, 5))
plt.xlabel("Grid position"); plt.ylabel("Podium rate"); plt.title("Podium rate by grid slot: home vs away")
plt.legend(["Away", "Home"]); plt.tight_layout(); plt.show()

## Question 2

In [ ]:
drivers_clean.head()

In [ ]:
races_clean.head()

In [ ]:
circuits_clean.head()

In [ ]:
results_clean.head()

In [ ]:
constructors_clean.head()

In [ ]:
races_drivers = drivers_clean.merge(results_clean, on="driverId", how="left").merge(races_clean, on="raceId", how="left")

In [ ]:
circuits_races = circuits_clean.merge(races_drivers, on="circuitId", how="left")

In [ ]:
circuits_races.to_csv('..\\ACL_9th-F1\\F1_clean\\' + 'circuits_races.csv', index=False)

In [ ]:
circuits_races.isnull().sum()

In [ ]:
circuits_races.drop(columns=["race_ms", "fastest_lap_number", "fastest_lap_rank", "fastest_lap_time_s", "fastest_lap_speed"], inplace=True)

In [ ]:
circuits_races.info()

In [ ]:
NAT_TO_COUNTRY = {
    'American': ['USA'], 'American-Italian': ['USA', 'Italy'],
    'Argentine': ['Argentina'], 'Argentine-Italian': ['Argentina', 'Italy'],
    'Argentinian': ['Argentina'], 'Australian': ['Australia'],
    'Austrian': ['Austria'], 'Belgian': ['Belgium'], 'Brazilian': ['Brazil'],
    'British': ['UK'], 'Canadian': ['Canada'], 'Chilean': ['Chile'],
    'Chinese': ['China'], 'Colombian': ['Colombia'], 'Czech': ['Czech Republic'],
    'Danish': ['Denmark'], 'Dutch': ['Netherlands'], 'East German': ['Germany'],
    'Finnish': ['Finland'], 'French': ['France'], 'German': ['Germany'],
    'Hungarian': ['Hungary'], 'Indian': ['India'], 'Indonesian': ['Indonesia'],
    'Irish': ['Ireland'], 'Italian': ['Italy'], 'Japanese': ['Japan'],
    'Liechtensteiner': ['Liechtenstein'], 'Malaysian': ['Malaysia'],
    'Mexican': ['Mexico'], 'Monegasque': ['Monaco'], 'New Zealander': ['New Zealand'],
    'Polish': ['Poland'], 'Portuguese': ['Portugal'], 'Rhodesian': ['Rhodesia'],
    'Russian': ['Russia'], 'South African': ['South Africa'], 'Spanish': ['Spain'],
    'Swedish': ['Sweden'], 'Swiss': ['Switzerland'], 'Thai': ['Thailand'],
    'Uruguayan': ['Uruguay'], 'Venezuelan': ['Venezuela'],
    'Hong Kong' : ['Hong Kong']
}

In [ ]:
def add_is_home(df, nationality_col='nationality', country_col='country'):
    """Add a 0/1 column 'is_home': 1 if the driver is racing in his own country."""
    df = df.copy()                                              # don't change the original table

    nat = df[nationality_col].str.strip()                       # fix 'Argentinian ' (trailing space)
    country = df[country_col].replace({'United States': 'USA'}) # Las Vegas uses 'United States'

    # Warn if a nationality is missing from the map, instead of failing silently
    unmapped = set(nat.dropna()) - set(NAT_TO_COUNTRY)
    if unmapped:
        print("Warning - nationalities not in the map (counted as not home):", unmapped)

    df['is_home'] = [int(c in NAT_TO_COUNTRY.get(n, []))
                     for n, c in zip(nat, country)]
    return df

In [ ]:
circuits_races = add_is_home(circuits_races,
                             nationality_col='drivers_nationality',
                             country_col='country',
                             out_col='driver_is_home')


In [ ]:
circuits_races.head()

In [ ]:
results_clean.head()

In [ ]:
indy_ids = races.loc[races['name'].str.contains('Indianapolis 500'), 'raceId']
circuits_races = circuits_races[~circuits_races['raceId'].isin(indy_ids)]

In [ ]:
def home_vs_away_at_grid(df, grid_position):
    """Show every driver who started from `grid_position`,
    split into home vs away, and print their podium rates."""
    subset = df[df['grid'] == grid_position]

    summary = (subset.groupby('is_home')['podium']
                     .agg(starts='count', podiums='sum', podium_rate='mean')
                     .rename(index={0: 'Away', 1: 'Home'}))
    summary['podium_rate'] = (summary['podium_rate'] * 100).round(1)

    print(f"Drivers starting from grid position {grid_position}:")
    print(summary, "\n")

    return subset[['raceId', 'driverId', 'drivers_nationality', 'country',
                   'is_home', 'grid', 'podium']]

In [ ]:
home_vs_away_drivers = home_vs_away_at_grid(circuits_races, 1)

## Observation: 

### Answering quesiton 2

- We have merged circuits, races, results and drivers
- Then we added a new column in the final table to check whether the driver is driving the race in his home country or not
- We also removed the 500 Indianopolis rows which was stated in the F1 document
- After that, we checked if every driver starting from the same grid position while comparing the is_home column which tells us if the driver is in his own country or not
- A chart was plotted to visualize the relation between the grid position and the podium rate while comparing is_home with the circuit country

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_home_effect(df, max_grid=10):
    """Bar chart: podium rate at home vs away, for each starting position."""
    d = df[(df['grid'] >= 1) & (df['grid'] <= max_grid)]   # grid 0 = pit lane, left out

    stats = (d.groupby(['grid', 'is_home'])['podium']
               .agg(rate='mean', n='count')
               .unstack('is_home'))
    rate = stats['rate'] * 100
    n = stats['n']
    # 95% error bars: how unsure we are, given how few drivers are in each group
    err = 1.96 * np.sqrt(stats['rate'] * (1 - stats['rate']) / n) * 100

    x = np.arange(len(rate))
    w = 0.4
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(x - w/2, rate[0], w, yerr=err[0], capsize=3, label='Away', color='#9AA5B1')
    ax.bar(x + w/2, rate[1], w, yerr=err[1], capsize=3, label='Home', color='#D1495B')

    # label how many home drivers are behind each red bar
    for i, g in enumerate(rate.index):
        ax.text(x[i] + w/2, rate.loc[g, 1] + err.loc[g, 1] + 1.5,
                f"n={n.loc[g, 1]}", ha='center', fontsize=8, color='#D1495B')

    ax.set_xticks(x)
    ax.set_xticklabels([f"P{g}" for g in rate.index])
    ax.set_xlabel('Starting grid position')
    ax.set_ylabel('Podium rate (%)')
    ax.set_title('Do drivers podium more at home? Podium rate by starting position')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()
    return stats

stats = plot_home_effect(circuits_races)

In [ ]:
circuits_races = circuits_races.merge(constructors_clean[['constructorId', 'constructor_nationality']], on='constructorId', how='left')

In [ ]:
circuits_races.head()

In [ ]:
def add_is_home(df, nationality_col='nationality', country_col='country', out_col='is_home'):
    """Add a 0/1 column: 1 if this nationality matches the circuit's country."""
    df = df.copy()
    nat = df[nationality_col].str.strip()
    country = df[country_col].replace({'United States': 'USA'})

    unmapped = set(nat.dropna()) - set(NAT_TO_COUNTRY)
    if unmapped:
        print("Warning - nationalities not in the map (counted as not home):", unmapped)

    df[out_col] = [int(c in NAT_TO_COUNTRY.get(n, [])) for n, c in zip(nat, country)]
    return df

In [ ]:
circuits_races = add_is_home(circuits_races,
                             nationality_col='constructor_nationality',   
                             country_col='country',
                             out_col='constructor_is_home')


In [ ]:
circuits_races.drop(columns=['is_home'], inplace=True)

In [ ]:
circuits_races.head()

In [ ]:
def home_vs_away_at_grid_constructor(df, grid_position, home_col='is_home'):
    """Compare podium rates of home vs away entries that started from the same grid spot."""
    subset = df[df['grid'] == grid_position]

    summary = (subset.groupby(home_col)['podium']
                     .agg(starts='count', podiums='sum', podium_rate='mean')
                     .rename(index={0: 'Away', 1: 'Home'}))
    summary['podium_rate'] = (summary['podium_rate'] * 100).round(1)

    print(f"Grid position {grid_position} ({home_col}):")
    print(summary, "\n")
    return subset

In [ ]:
constructors_home_vs_away = home_vs_away_at_grid_constructor(circuits_races, 1, home_col='constructor_is_home')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_home_effect(df, home_col='constructor_is_home', who='Constructors', max_grid=10):
    """Bar chart of podium rate at home vs away, for each starting position."""
    d = df[(df['grid'] >= 1) & (df['grid'] <= max_grid)]   # grid 0 = pit lane, left out

    stats = (d.groupby(['grid', home_col])['podium']
               .agg(rate='mean', n='count')
               .unstack(home_col))
    rate = stats['rate'] * 100
    n = stats['n']
    err = 1.96 * np.sqrt(stats['rate'] * (1 - stats['rate']) / n) * 100   # 95% error bars

    x = np.arange(len(rate))
    w = 0.4
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(x - w/2, rate[0], w, yerr=err[0], capsize=3, label='Away', color='#9AA5B1')
    ax.bar(x + w/2, rate[1], w, yerr=err[1], capsize=3, label='Home', color='#D1495B')

    for i, g in enumerate(rate.index):
        ax.text(x[i] + w/2, rate.loc[g, 1] + err.loc[g, 1] + 1.5,
                f"n={n.loc[g, 1]}", ha='center', fontsize=8, color='#D1495B')

    ax.set_xticks(x)
    ax.set_xticklabels([f"P{g}" for g in rate.index])
    ax.set_xlabel('Starting grid position')
    ax.set_ylabel('Podium rate (%)')
    ax.set_title(f'{who}: podium rate at home vs away, by starting position')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()
    return stats

In [ ]:
[col for col in circuits_races.columns if 'home' in col]

In [ ]:
constructor_stats = plot_home_effect(circuits_races, home_col='constructor_is_home', who='Constructors')

### We have also compared constructor nationalities and plotted the relation between its drivers' starting position and the podium rate

## Question 3

In [ ]:
status.head()

In [ ]:
status_race = results_clean.merge(status, left_on='statusId_race', right_on='statusId', how='left')

In [ ]:
status_race.head()

In [ ]:
MECHANICAL = {
    # engine / power unit
    'Engine', 'Power Unit', 'ERS', 'Turbo', 'Supercharger', 'Battery', 'Power loss',
    'Engine fire', 'Engine misfire', 'Ignition', 'Spark plugs', 'Injection',
    'Distributor', 'Magneto', 'Alternator', 'Crankshaft',
    # gearbox / drivetrain
    'Gearbox', 'Transmission', 'Clutch', 'Differential', 'Driveshaft',
    'Halfshaft', 'CV joint', 'Axle', 'Drivetrain', 'Launch control',
    # fluids / cooling / fuel
    'Hydraulics', 'Oil leak', 'Oil pressure', 'Oil pump', 'Oil pipe', 'Oil line',
    'Water leak', 'Water pressure', 'Water pump', 'Water pipe', 'Radiator',
    'Overheating', 'Cooling system', 'Fuel pressure', 'Fuel pump', 'Fuel system',
    'Fuel leak', 'Fuel pipe', 'Fuel', 'Out of fuel', 'Refuelling', 'Fuel rig',
    # chassis / electrics / other car parts
    'Electrical', 'Electronics', 'Suspension', 'Brakes', 'Brake duct', 'Steering',
    'Throttle', 'Exhaust', 'Pneumatics', 'Vibrations', 'Mechanical', 'Technical',
    'Chassis', 'Handling', 'Track rod', 'Wheel bearing', 'Wheel rim',
    'Heat shield fire', 'Fire', 'Seat', 'Driver Seat', 'Safety belt',
}

In [ ]:
def add_mechanical_flag(df, status_col='status'):
    """Add a 0/1 column: 1 if the driver retired because the car broke."""
    df = df.copy()
    df['is_mechanical_dnf'] = df[status_col].isin(MECHANICAL).astype(int)
    return df

In [ ]:
status_race = add_mechanical_flag(status_race)

# see which drivers had a mechanical failure
status_race[status_race['is_mechanical_dnf'] == 1].head()

In [ ]:
status_race = status_race.merge(races_clean[['raceId', 'race_year', 'race_round']], on='raceId', how='left')

In [ ]:
def mechanical_rate_by_era(df, min_starts=30):
    """Mechanical DNF rate (%) per constructor for 2014-2021 and 2022-2024."""
    not_started = {'Withdrew', 'Did not qualify', 'Did not prequalify', '107% Rule',
                   'Excluded', 'Not restarted'}

    d = (df.merge(races[['raceId', 'year']], on='raceId', how='left')
           .merge(constructors[['constructorId', 'name']], on='constructorId', how='left'))
    d = d[(d['year'] >= 2014) & ~d['status'].isin(not_started)]
    d['era'] = np.where(d['year'] <= 2021, '2014-2021', '2022-2024')

    table = (d.groupby(['era', 'name'])['is_mechanical_dnf']
               .agg(starts='size', mechanical_dnfs='sum'))
    table['rate_%'] = (table['mechanical_dnfs'] / table['starts'] * 100).round(1)

    return table[table['starts'] >= min_starts]      # drop teams with too few starts

rates = mechanical_rate_by_era(status_race)
for era in ['2014-2021', '2022-2024']:
    print(era)
    print(rates.loc[era].sort_values('rate_%', ascending=False).head(5), "\n")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_mechanical_rates(df, rates, top_n=10):
    """Side-by-side bar charts: top constructors by mechanical-retirement rate, per era."""
    not_started = {'Withdrew', 'Did not qualify', 'Did not prequalify', '107% Rule',
                   'Excluded', 'Not restarted'}

    # overall average per era (all constructors), used as the reference line
    d = df.merge(races[['raceId', 'year']], on='raceId', how='left')
    d = d[(d['year'] >= 2014) & ~d['status'].isin(not_started)]
    d['era'] = np.where(d['year'] <= 2021, '2014-2021', '2022-2024')
    era_avg = d.groupby('era')['is_mechanical_dnf'].mean() * 100

    eras = ['2014-2021', '2022-2024']
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True)   # same x-scale so eras are comparable

    for ax, era, color in zip(axes, eras, ['#D1495B', '#2E86AB']):
        top = rates.loc[era].sort_values('rate_%').tail(top_n)
        bars = ax.barh(top.index, top['rate_%'], color=color)

        # label each bar with its rate and the raw counts behind it
        for bar, (_, row) in zip(bars, top.iterrows()):
            ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                    f"{row['rate_%']:.1f}%  ({int(row['mechanical_dnfs'])}/{int(row['starts'])})",
                    va='center', fontsize=8)

        ax.axvline(era_avg[era], color='black', linestyle='--', linewidth=1)
        ax.text(era_avg[era] + 0.3, -0.9, f"era average {era_avg[era]:.1f}%", fontsize=8)
        ax.set_title(era, fontsize=12)
        ax.set_xlabel('Mechanical retirements per start (%)')
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Constructors with the highest mechanical-retirement rate', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_mechanical_rates(status_race, rates)

### Observation — Mechanical-retirement rate by constructor (2014–2021 vs 2022–2024)

**What we did**
1. Merged `results` with `status` to get the reason each driver's race ended.
2. Classified each status into a category. Only failures of the car itself (e.g. *Engine, Power Unit,
   Gearbox, Hydraulics, Brakes, ERS, Suspension*) were counted as **mechanical**. Crashes (*Accident,
   Collision*), contact damage (*Collision damage, Puncture, Front wing*) and unclear statuses
   (*Retired, Wheel nut, Tyre*) were excluded, because they are not caused by the car failing on its own.
3. Removed entries that never started the race (*Withdrew, Did not qualify, Excluded*).
4. Split the data into two eras: **2014–2021** (first hybrid-engine era) and **2022–2024** (new
   ground-effect regulations).
5. Computed a **rate**, not a count: mechanical retirements ÷ race starts per constructor, so teams with
   more races are compared fairly. Teams with fewer than 30 starts in an era were left out, since a few
   failures would give them misleadingly extreme rates.

**Findings**
- **2014–2021:** Caterham (23.5%), Lotus F1 (17.3%) and Toro Rosso (16.1%) had the highest rates,
  followed by Renault (14.0%) and McLaren (13.8%). Many of these were customer or works teams of
  struggling engine suppliers early in the hybrid era (e.g. McLaren–Honda 2015–2017).
- **2022–2024:** Alfa Romeo (12.5%) and Alpine (11.0%) were clearly the least reliable; most other teams
  were around 4–6%.
- **Reliability improved across the grid:** the average mechanical-retirement rate fell from **9.3%**
  to **5.7%** of starts. The hybrid power units matured, so a car breaking down became much less common.

**Limitations**
- Teams changed names between eras (Toro Rosso → AlphaTauri → RB, Sauber → Alfa Romeo → Sauber,
  Renault → Alpine, Force India → Racing Point → Aston Martin). Each name has its own `constructorId`,
  so they are shown as separate teams.
- The mechanical/non-mechanical split is a judgement call. Generic statuses like *Retired* give no
  reason and were excluded, which may slightly undercount mechanical failures.
- 2022–2024 is only 3 seasons (~136 starts per team), so rates there rest on small counts
  (e.g. 15 failures for Alpine) and are less certain than the 2014–2021 figures.

In [ ]:
FAILURE_GROUP = {
    'Power unit': ['Engine', 'Power Unit', 'ERS', 'Turbo', 'Supercharger', 'Battery',
                   'Power loss', 'Engine fire', 'Engine misfire', 'Ignition', 'Spark plugs',
                   'Injection', 'Distributor', 'Magneto', 'Alternator', 'Crankshaft'],
    'Gearbox / drivetrain': ['Gearbox', 'Transmission', 'Clutch', 'Differential', 'Driveshaft',
                             'Halfshaft', 'CV joint', 'Axle', 'Drivetrain', 'Launch control'],
    'Hydraulics': ['Hydraulics', 'Pneumatics'],
    'Cooling / fluids / fuel': ['Oil leak', 'Oil pressure', 'Oil pump', 'Oil pipe', 'Oil line',
                                'Water leak', 'Water pressure', 'Water pump', 'Water pipe',
                                'Radiator', 'Overheating', 'Cooling system', 'Fuel pressure',
                                'Fuel pump', 'Fuel system', 'Fuel leak', 'Fuel pipe', 'Fuel',
                                'Out of fuel', 'Refuelling', 'Fuel rig'],
    'Electrical': ['Electrical', 'Electronics'],
    'Brakes': ['Brakes', 'Brake duct'],
    'Suspension / steering': ['Suspension', 'Steering', 'Track rod', 'Wheel bearing',
                              'Wheel rim', 'Handling', 'Chassis'],
    'Other mechanical': ['Throttle', 'Exhaust', 'Vibrations', 'Mechanical', 'Technical',
                         'Heat shield fire', 'Fire', 'Seat', 'Driver Seat', 'Safety belt'],
}

# flip it: status -> failure type  (e.g. 'Engine' -> 'Power unit')
STATUS_TO_GROUP = {status: group for group, statuses in FAILURE_GROUP.items() for status in statuses}

# safety check: every mechanical status has exactly one group
assert set(STATUS_TO_GROUP) == MECHANICAL, set(MECHANICAL) ^ set(STATUS_TO_GROUP)

In [ ]:
def get_mechanical_failures(df):
    """Return only mechanical retirements (2014+), with era, team name and failure type."""
    d = (df.merge(races[['raceId', 'year']], on='raceId', how='left')
           .merge(constructors[['constructorId', 'name']], on='constructorId', how='left'))
    d = d[(d['year'] >= 2014) & (d['is_mechanical_dnf'] == 1)].copy()
    d['era'] = np.where(d['year'] <= 2021, '2014-2021', '2022-2024')
    d['failure_type'] = d['status'].map(STATUS_TO_GROUP)
    return d

mech = get_mechanical_failures(status_race)

failure_counts = pd.crosstab(mech['failure_type'], mech['era'])
failure_counts

In [ ]:
def plot_failure_types(failure_counts):
    """Share of mechanical retirements caused by each failure type, per era."""
    share = failure_counts / failure_counts.sum() * 100        # % within each era
    share = share.loc[share.sum(axis=1).sort_values().index]   # biggest at the top

    y = np.arange(len(share))
    h = 0.4
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(y + h/2, share['2014-2021'], h, color='#D1495B',
            label=f"2014-2021 (n={failure_counts['2014-2021'].sum()})")
    ax.barh(y - h/2, share['2022-2024'], h, color='#2E86AB',
            label=f"2022-2024 (n={failure_counts['2022-2024'].sum()})")

    for i, f in enumerate(share.index):
        ax.text(share.loc[f, '2014-2021'] + 0.5, i + h/2, f"{share.loc[f, '2014-2021']:.0f}%", va='center', fontsize=8)
        ax.text(share.loc[f, '2022-2024'] + 0.5, i - h/2, f"{share.loc[f, '2022-2024']:.0f}%", va='center', fontsize=8)

    ax.set_yticks(y)
    ax.set_yticklabels(share.index)
    ax.set_xlabel('Share of all mechanical retirements in the era (%)')
    ax.set_title('Which mechanical failures caused the most retirements?')
    ax.legend(loc='lower right')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_failure_types(failure_counts)

### Observation — Which failure types caused the most mechanical retirements?

**What the plot shows:** each bar is the share (%) of all mechanical retirements in that era caused by
one failure type. Shares are used instead of raw counts because 2014–2021 covers 8 seasons
(302 failures) while 2022–2024 covers only 3 (77 failures).

**Findings**
- **Power unit failures were the #1 cause in both eras** (40% → 42%). Even as engines matured,
  the power unit stayed the most fragile part of the car.
- **Cooling / fluids / fuel failures rose sharply**, from 9% to 25%, becoming the second-largest
  cause in 2022–2024.
- **Brakes (15% → 5%), suspension/steering (8% → 3%) and electrical (6% → 1%) failures became
  much rarer**, showing that these parts became far more reliable.
- **Gearbox / drivetrain failures stayed stable** at about 13–14%.

**Note:** the 2022–2024 shares are based on only 77 failures, so a few extra failures in one
category can shift its percentage noticeably.